In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/heart.csv")
X = df.drop(columns="target")
y = df["target"]
print(X.shape, y.shape)

(303, 13) (303,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("train:", X_train.shape, "| test:", X_test.shape)
print("نرخ بیماری train:", round(y_train.mean(), 3))
print("نرخ بیماری test :", round(y_test.mean(), 3))

train: (242, 13) | test: (61, 13)
نرخ بیماری train: 0.459
نرخ بیماری test : 0.459


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

In [5]:
preprocess.fit(X_train)
Xt_train = preprocess.transform(X_train)
Xt_test = preprocess.transform(X_test)

print("train:", Xt_train.shape, "| test:", Xt_test.shape)
print("NaN باقی‌مانده:", np.isnan(Xt_train).sum(), np.isnan(Xt_test).sum())

n = len(num_cols)
print("میانگین ۵ ستون عددی train:", Xt_train[:, :n].mean(axis=0).round(3))
print("انحراف معیار train      :", Xt_train[:, :n].std(axis=0).round(3))
print("میانگین ۵ ستون عددی test :", Xt_test[:, :n].mean(axis=0).round(3))

train: (242, 28) | test: (61, 28)
NaN باقی‌مانده: 0 0
میانگین ۵ ستون عددی train: [-0. -0.  0.  0. -0.]
انحراف معیار train      : [1. 1. 1. 1. 1.]
میانگین ۵ ستون عددی test : [-0.061  0.206 -0.296 -0.078  0.179]


In [6]:
print("میانه‌های یادگرفته‌شده:",
      dict(zip(num_cols, preprocess.named_transformers_["num"]["imputer"].statistics_)))
print("پرتکرارترین‌ها:",
      dict(zip(cat_cols, preprocess.named_transformers_["cat"]["imputer"].statistics_)))
print("نام ستون‌های نهایی:", list(preprocess.get_feature_names_out())[:8], "...")

میانه‌های یادگرفته‌شده: {'age': np.float64(56.0), 'trestbps': np.float64(130.0), 'chol': np.float64(244.5), 'thalach': np.float64(153.5), 'oldpeak': np.float64(0.8)}
پرتکرارترین‌ها: {'sex': np.float64(1.0), 'cp': np.float64(4.0), 'fbs': np.float64(0.0), 'restecg': np.float64(0.0), 'exang': np.float64(0.0), 'slope': np.float64(1.0), 'ca': np.float64(0.0), 'thal': np.float64(3.0)}
نام ستون‌های نهایی: ['num__age', 'num__trestbps', 'num__chol', 'num__thalach', 'num__oldpeak', 'cat__sex_0.0', 'cat__sex_1.0', 'cat__cp_1.0'] ...
